# 02 — Phân loại cảm xúc bình luận (NLP cổ điển)

## Mục tiêu
- Biến text thành số bằng **TF-IDF** (không dùng LLM).
- Train classifier (Logistic Regression) để gắn nhãn positive / negative / neutral.
- Chuẩn bị tư duy cho embedding ở bài RAG (vẫn là "text → vector", nhưng cách khác).

## Kiến thức cần nhớ
- Tabular ML dùng cột số; NLP cổ điển cần **vector hóa** câu trước.
- TF-IDF: từ xuất hiện nhiều trong 1 câu nhưng ít trong toàn corpus → quan trọng hơn.
- Accuracy / F1 đo chất lượng phân loại.

Dataset: `data/binh_luan.csv`.


## 1. Load dữ liệu


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data/binh_luan.csv")
df = pd.read_csv(DATA)
print(df.shape)
print(df["label"].value_counts())
df.head()


## 2. Train/test + TF-IDF + Logistic Regression


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)

clf = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ("lr", LogisticRegression(max_iter=1000)),
    ]
)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))


## 3. Thử vài câu mới


In [ ]:
mau = [
    "San pham tot qua, se mua lai",
    "That vong, hang loi",
    "Tam on, khong co gi dac biet",
]
for c in mau:
    print(c, "->", clf.predict([c])[0])


## 4. Lưu model để FastAPI dùng


In [ ]:
from pathlib import Path

MODEL_PATH = Path("../../data/sentiment_model.joblib")
joblib.dump(clf, MODEL_PATH)
print("Saved:", MODEL_PATH.resolve())


## 5. Gọi API (sau khi chạy uvicorn)

Trong terminal:

```powershell
cd notebooks\02_sentiment
uvicorn api_predict:app --reload
```

Rồi test (terminal khác hoặc cell dưới với `httpx`/`requests`).


In [ ]:
# Tuy chon: goi API neu dang chay
import json

try:
    import urllib.request

    body = json.dumps({"text": "Chat luong kem"}).encode("utf-8")
    req = urllib.request.Request(
        "http://127.0.0.1:8000/predict",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=2) as resp:
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("API chua chay hoac loi:", type(e).__name__, e)
    print("Bo qua cell nay neu ban chua mo uvicorn.")


## Bài tập nhỏ

1. Thêm 3 câu bình luận mới vào CSV (hoặc list), train lại, xem F1 có đổi không.
2. Đổi `LogisticRegression` sang `MultinomialNB` (`sklearn.naive_bayes`) và so sánh accuracy.
3. Giải thích khác biệt TF-IDF vs embedding (bài RAG) bằng 2–3 câu.


In [ ]:
# TODO: bai tap
# from sklearn.naive_bayes import MultinomialNB


## Checklist — đã hiểu khi…
- [ ] Nói được vì sao text cần vector hóa trước khi đưa vào model
- [ ] Đọc được accuracy / F1 từ `classification_report`
- [ ] Lưu được model bằng `joblib`
- [ ] Biết chạy `uvicorn api_predict:app --reload`
